# Datamine ASTRAN Process Example

This notebook demonstrates how to configure and run the **`astran`** process wrapper in `dmstudio`.

## Process Description

## ASTRAN

This process reads one or more ASCII sample assay files in SIF format and updates a DATAMINE database file by matching sample numbers. Up to 10 grade fields may be updated from each file.

**ASTRAN** expects to find a file **ASTRAN.LST** containing a list of SIF files to be processed. This file may be prepared by an operating system pre-processing script. Any files that have all records matched are renamed.

SIF files that do not have all their samples matched will be rewritten with a flag 'MATCHED: date' at the end of any matched records. On subsequent runs these previously matched records may be ignored or rematched according to the value of the @**UPDATE** parameter.

A database file &**XREF** is used to cross reference assay names as they appear in the SIF files (**ASSNAM**) and the database field name (**ELEMENT**). Multiple values of **ASSNAM** may be referenced to a field. This file also has a units field which specifies the units of the assay as stored in the database. Assays in the SIF file will be converted to match these units. Only one value of units should appear in &**XREF** for each value of **ELEMENT**.

Example of &**XREF** :

## ELEMENT UNITS ASSNAM

AU1 ppm Au
AU2 ppm Au(R)
AU2 ppm Au(R)

## SULPHUR % S

A 2 valued DATAMINE environment variable **ASTRAN_COMMAND** is used by **ASTRAN**.

The first value is an operating system command or script to be run to pre-process the ASCII files and produce the ASTRAN.LST file.

The second value gives the format of the command to be used to rename a fully matched file. Each individual file name will be substituted for the $1.

### Input Files:

* **in**:
  Input database file.
  Required=No

* **xref**:
  Required assay name cross-reference file. This file is used to link the names of assay
  fields in the database file to fields in the assay transfer {SIF} file. It is also used
  to convert results from the assay file to database units. Required fields in the XREF
  file are:
  Required=No

### Output Files:

### Fields:

* **sampleid** (Any : IN):
  Optional name of sample identifier field in the IN file. Only required if the name of
  sample identifier field is not "SAMPLEID".
  Default=SAMPLEID
  Required=No

### Parameters:

* **sprefix**:
  Optional parameter to specify number of prefix characters. If "SAMPLEID" is numeric,
  this must be 0 if specified, otherwise it must be less than 11. (2)
  Range=0,11
  Values=0,1,2,3,4,5,6,7,8,9,10,11
  Default=2
  Required=No

* **sdigits**:
  Optional parameter to specify number of digits to form numeric portion of "SAMPLEID".
  If "SAMPLEID" is numeric then SDIGIT must lie between 1 and 6. If "SAMPLEID" is alpha-
  numeric, SDIGIT must lie between 0 and 16. (6)
  Range=0,16
  Values=0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
  Default=6
  Required=No

* **maxerrs**:
  Maximum number of errors that will be tolerated before processing is aborted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **update**:
  Controls treatment of previously updated samples. Options: (0): Ignore previously
  updated samples.; 1: Check previously updated samples but only update if the assay field
  has a missing value.; 2: As 1 but overwrite any value in the assay field.
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('astran')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute astran
print("Running astran...")
dm_cmd.astran(
    # in_i='t_assays',  # optional
    # xref_i='optional',  # optional
    # sampleid_f='optional',  # optional
    # sprefix_p=2,  # optional
    # sdigits_p=6,  # optional
    # maxerrs_p='optional',  # optional
    # update_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("astran execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_astran_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")